In [10]:
# ── CELL 1: Install Dependencies ───────────────────────────────────────────────

%%capture
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets


In [11]:

# ── CELL 2: Imports & Config ───────────────────────────────────────────────────

import json
import torch
import random
from datasets import Dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig

# ── Hyperparameters ────────────────────────────────────────────────────────────
MODEL_NAME      = "unsloth/Qwen2.5-Coder-3B-Instruct"
MAX_SEQ_LENGTH  = 2048
LORA_R          = 16
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.05      # ↑ from 0.0 — mild regularization

BATCH_SIZE      = 2
GRAD_ACCUM      = 4         # effective batch = 8
LEARNING_RATE   = 2e-4
NUM_EPOCHS      = 2         # ↓ from 3 — more data means less re-exposure needed
WARMUP_STEPS    = 50
VAL_SPLIT       = 0.1
SEED            = 42

OUTPUT_DIR      = "./qwen_combined_checkpoints"



In [12]:

# ── CELL 3: Load Dataset ───────────────────────────────────────────────────────

import os

# Try sidebar first, fall back to upload
candidate_paths = [
    "/content/combined_personality_finetune.jsonl",
    "/content/sample_data/combined_personality_finetune.jsonl",
]
jsonl_path = next((p for p in candidate_paths if os.path.exists(p)), None)

if jsonl_path is None:
    from google.colab import files
    print("📂 Upload combined_personality_finetune.jsonl ...")
    uploaded = files.upload()
    jsonl_path = next((k for k in uploaded if k.endswith(".jsonl")), None)
    if not jsonl_path:
        raise FileNotFoundError("No .jsonl file found in upload")

print(f"✅ Loaded: {jsonl_path}")

with open(jsonl_path) as f:
    raw_examples = [json.loads(l) for l in f]

print(f"   Total examples: {len(raw_examples)}")
neg = sum(1 for e in raw_examples if "[NEGATIVE]" in e["conversations"][0]["content"])
pos = sum(1 for e in raw_examples if "[POSITIVE]" in e["conversations"][0]["content"])
print(f"   NEGATIVE: {neg}")
print(f"   POSITIVE: {pos}")


✅ Loaded: /content/combined_personality_finetune.jsonl
   Total examples: 4292
   NEGATIVE: 1883
   POSITIVE: 2409


In [13]:

# ── CELL 4: Load Model ─────────────────────────────────────────────────────────

print(f"\n🤖 Loading {MODEL_NAME} with 4-bit QLoRA...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,
    load_in_4bit    = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    target_modules             = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = SEED,
)

# Force correct Qwen2.5 tokens
tokenizer.eos_token = "<|im_end|>"
tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Model loaded")
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")




🤖 Loading unsloth/Qwen2.5-Coder-3B-Instruct with 4-bit QLoRA...
==((====))==  Unsloth 2026.6.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

✅ Model loaded
   Trainable params: 29,933,568


In [14]:
# ── CELL 5: Format Dataset ─────────────────────────────────────────────────────

SYSTEM_PROMPT = (
    "You are an AI tutoring assistant for an Intelligent Tutoring System. "
    "When given a [NEGATIVE] prompt, generate Python code that contains bugs "
    "appropriate to the specified student persona or bug type. "
    "When given a [POSITIVE] prompt, generate correct, clean Python code."
)

def format_example(example):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["conversations"][0]["content"]},
        {"role": "assistant", "content": example["conversations"][1]["content"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize              = False,
        add_generation_prompt = False,
    )
    return {"text": text}

print("🔧 Formatting dataset with Qwen2.5 chat template...")

random.seed(SEED)
shuffled = raw_examples.copy()
random.shuffle(shuffled)

split_idx  = int(len(shuffled) * (1 - VAL_SPLIT))
train_data = shuffled[:split_idx]
val_data   = shuffled[split_idx:]

print(f"   Train: {len(train_data)} examples")
print(f"   Val:   {len(val_data)} examples")

train_dataset = Dataset.from_list(train_data).map(format_example)
val_dataset   = Dataset.from_list(val_data).map(format_example)

print("\n── Sample formatted example (truncated) ──")
print(train_dataset[0]["text"][:600])
print("...")

🔧 Formatting dataset with Qwen2.5 chat template...
   Train: 3862 examples
   Val:   430 examples


Map:   0%|          | 0/3862 [00:00<?, ? examples/s]

Map:   0%|          | 0/430 [00:00<?, ? examples/s]


── Sample formatted example (truncated) ──
<|im_start|>system
You are an AI tutoring assistant for an Intelligent Tutoring System. When given a [NEGATIVE] prompt, generate Python code that contains bugs appropriate to the specified student persona or bug type. When given a [POSITIVE] prompt, generate correct, clean Python code.<|im_end|>
<|im_start|>user
[POSITIVE] Generate a correct Python function for the following problem.

Problem: Write a function to find the length of the longest sub-sequence such that elements in the subsequences are consecutive integers.<|im_end|>
<|im_start|>assistant
def find_longest_conseq_subseq(arr, n):
  
...


In [15]:
# ── CELL 6: Configure Trainer ──────────────────────────────────────────────────

steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
total_steps     = steps_per_epoch * NUM_EPOCHS
eval_steps      = max(steps_per_epoch // 4, 20)   # eval 4x per epoch — catch divergence early

print(f"\n📋 Training plan:")
print(f"   Steps per epoch : {steps_per_epoch}")
print(f"   Total steps     : {total_steps}")
print(f"   Eval every      : {eval_steps} steps")

# Force EOS/PAD again in case kernel state is stale
tokenizer.eos_token = "<|im_end|>"
tokenizer.pad_token = tokenizer.eos_token

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    eos_token                   = "<|im_end|>",
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    warmup_steps                = WARMUP_STEPS,
    lr_scheduler_type           = "cosine",
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = 10,
    eval_strategy               = "steps",
    eval_steps                  = eval_steps,
    save_strategy               = "steps",
    save_steps                  = eval_steps,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    dataset_text_field          = "text",
    packing                     = False,
    seed                        = SEED,
    report_to                   = "none",
)

trainer = SFTTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = train_dataset,
    eval_dataset     = val_dataset,
    args             = training_args,
)

# Only train on assistant responses (NAT — loss computed only on generated code)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part    = "<|im_start|>assistant\n",
)

print("✅ Trainer configured")





📋 Training plan:
   Steps per epoch : 482
   Total steps     : 964
   Eval every      : 120 steps


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3862 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/430 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=6):   0%|          | 0/3862 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/3862 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/430 [00:00<?, ? examples/s]

Filter (num_proc=6):   0%|          | 0/430 [00:00<?, ? examples/s]

✅ Trainer configured


In [16]:
# ── CELL 7: Train ──────────────────────────────────────────────────────────────

print("\n🚀 Starting training...")
print("   Watch eval_loss — if it rises while train_loss falls = overfitting")
print()

trainer_stats = trainer.train()

print(f"\n✅ Training complete!")
print(f"   Runtime         : {trainer_stats.metrics['train_runtime']:.0f}s")
print(f"   Final train loss: {trainer_stats.metrics['train_loss']:.4f}")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.



🚀 Starting training...
   Watch eval_loss — if it rises while train_loss falls = overfitting



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,862 | Num Epochs = 2 | Total steps = 966
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
120,0.369969,0.354575
240,0.257863,0.278847
360,0.301807,0.243952
480,0.225273,0.215128
600,0.117710,0.201557
720,0.136666,0.188267
840,0.116584,0.182454
960,0.122955,0.181082
966,0.122955,0.181084


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


✅ Training complete!
   Runtime         : 4317s
   Final train loss: 0.2163


In [17]:

# ── CELL 8: Test Inference ─────────────────────────────────────────────────────
# Generate examples for each persona + a bug-type prompt for comparison

FastLanguageModel.for_inference(model)

def generate(prompt: str, max_new_tokens: int = 512) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs,
            max_new_tokens = max_new_tokens,
            temperature    = 0.7,
            top_p          = 0.9,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)


TEST_PROBLEM = "Write a Python function to check if a number is prime."

# Persona-conditioned generations
PERSONALITY_PROMPTS = {
    "CONFUSED_STUDENT":
        "a confused student who misunderstands the problem requirements "
        "and writes code that doesn't quite match what's being asked",
    "SYNTAX_STRUGGLER":
        "a student who struggles with Python syntax and makes errors like "
        "wrong indentation, missing colons, or incorrect operator usage",
    "IMPATIENT_STUDENT":
        "an impatient student who writes code quickly without thinking it "
        "through, often missing edge cases or making off-by-one errors",
    "OVERCONFIDENT_WRONG":
        "an overconfident student who writes code that looks polished and "
        "complete but contains fundamental logical errors",
}

for persona, desc in PERSONALITY_PROMPTS.items():
    print(f"\n── {persona} ─────────────────────────────────────────")
    p = (f"[NEGATIVE] Generate a Python function for the following problem "
         f"as it would be written by {desc}.\n\nProblem: {TEST_PROBLEM}")
    print(generate(p))

print(f"\n── POSITIVE (correct) ──────────────────────────────────────")
p = (f"[POSITIVE] Generate a correct Python function for the following problem."
     f"\n\nProblem: {TEST_PROBLEM}")
print(generate(p))




The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



── CONFUSED_STUDENT ─────────────────────────────────────────


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

def prime_num(num):
    if num <= 1:
        return False
    for i in range(2, num):
        if num % i == 0:
            return False
    return True

── SYNTAX_STRUGGLER ─────────────────────────────────────────


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


def prime_num(n):
    if n <= 1:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

── IMPATIENT_STUDENT ─────────────────────────────────────────


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


def prime_num(num):
    if num <= 1:
        return False
    for i in range(2, int(num**0.5) + 1):
        if num % i == 0:
            return False
    return True

── OVERCONFIDENT_WRONG ─────────────────────────────────────────


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


def prime_num(n):
    if n <= 1:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

── POSITIVE (correct) ──────────────────────────────────────
def prime_num(num):
    if num < 2:
        return False
    for i in range(2, int(num**0.5) + 1):
        if num % i == 0:
            return False
    return True


In [18]:
# ── CELL 9: Push to HuggingFace ────────────────────────────────────────────────

from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))   # uses Colab secret

REPO_NAME = "madane007/qwen-its-combined"   # change if you want a different name

model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

print(f"\n✅ Pushed to https://huggingface.co/{REPO_NAME}")

README.md:   0%|          | 0.00/575 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  558kB /  120MB            

Saved model to https://huggingface.co/madane007/qwen-its-combined


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpv2wqx5eo/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpv2wqx5eo/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            


✅ Pushed to https://huggingface.co/madane007/qwen-its-combined
